In [8]:
import cv2
import numpy as np

### Ejercicio 1

Implementa una función `ejercicio1(imagen)` que reciba una imagen en escala de grises y aplique una transformación de intensidad definida a trozos. Para ello:

1. Normaliza la imagen al rango $[0,1]$.
2. Genera dos números aleatorios $r_1$ y $r_2$ en el intervalo $[0,1]$, de forma que $r_1 < r_2$ (si no se cumple, intercámbialos).
3. Aplica la siguiente transformación $T(x)$ a cada intensidad $x$ de la imagen:

$$
T(x) =
\begin{cases}
\sqrt{r_1 \cdot x} & \text{si } x \le r_1 \\[2mm]
r_1 + (r_2 - r_1)\left(\dfrac{x - r_1}{r_2 - r_1}\right)^2 & \text{si } r_1 < x \le r_2 \\[2mm]
r_2 + (1 - r_2)\left(\dfrac{x - r_2}{1 - r_2}\right)^{1/2} & \text{si } x > r_2
\end{cases}
$$

4. Devuelve la imagen transformada en formato `uint8` (rango $[0,255]$).

Aplica la función sobre la imagen `arbol.jpg` y muestra la imagen original y la transformada. ¿Qué efecto tiene cada uno de los tramos sobre las intensidades de la imagen?

In [9]:
def ejercicio1(imagen):
    imagen = np.float32(imagen)
    r1 = np.random.rand()
    r2 = np.random.rand()
    if r1 > r2:
        r1,r2 =  r2,r1
    imagen_norm = imagen/255
    imagen_transformada = np.zeros_like(imagen_norm)
    
    imagen_transformada[imagen_norm <= r1] = (imagen_norm[imagen_norm <= r1] * r1) ** 0.5
    imagen_transformada[(imagen_norm > r1) & (imagen_norm <= r2)] = (r1 + (r2 - r1) * ((imagen_norm[(imagen_norm > r1) & (imagen_norm <= r2)] - r1) / (r2 - r1)) ** 2)
    imagen_transformada[imagen_norm > r2] = (r2 + (1 - r2) * ((imagen_norm[imagen_norm > r2] - r2) / (1 - r2)) ** 0.5)

    imagen_transformada = np.uint8(imagen_transformada * 255) 
    return imagen_transformada


In [10]:
imagen = cv2.imread('arbol.jpg', cv2.IMREAD_GRAYSCALE)

# Aplicar el filtro no homogéneo
imagen2 =  ejercicio1(imagen)

# Mostrar la imagen original y la imagen filtrada
cv2.imshow('original', imagen)
cv2.imshow('ampliacion', imagen2)

# Esperar a que el usuario presione una tecla para cerrar las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()

### Ejercicio 2

Implementa una función `ejercicio2(imagen, Smax)` que aplique un filtro no lineal y no homogéneo. La función recibe una imagen en escala de grises y un tamaño máximo de ventana `Smax` (impar).

Para cada píxel $(i,j)$ de la imagen:

1. Recorre ventanas centradas en el píxel de tamaño $3\times3$, $5\times5$, ..., $S_{max}\times S_{max}$.
2. En cada ventana calcula la media $\mu$ y la desviación típica $\sigma$ de sus intensidades.
3. Elige un valor aleatorio dentro del intervalo $\left[\mu - \dfrac{\sigma}{2},\; \mu + \dfrac{\sigma}{2}\right]$. Para ello, genera un número aleatorio $\alpha \in [0,1]$ y calcula:
$$
v = \left(\mu - \frac{\sigma}{2}\right) + \alpha\left[\left(\mu + \frac{\sigma}{2}\right) - \left(\mu - \frac{\sigma}{2}\right)\right]
$$
4. El nuevo valor del píxel será la media de los valores $v$ obtenidos en todas las ventanas.

Para no perder filas ni columnas, amplía la imagen antes de filtrar usando `cv2.copyMakeBorder` con `cv2.BORDER_REPLICATE`. La imagen resultante debe tener el mismo tamaño que la original y estar en formato `uint8`.

Aplica la función sobre la imagen `arbol.jpg` con `Smax = 5` y muestra la imagen original y la filtrada.

In [13]:
def ejercicio2(imagen,Smax):
    imagen = np.float32(imagen)
    bordeMax = Smax//2
    filas,col = imagen.shape
    imagen_ampliada = cv2.copyMakeBorder(imagen,bordeMax,bordeMax,bordeMax,bordeMax,cv2.BORDER_REPLICATE)
    imagen_filtrada = np.zeros((filas,col))
    res = 0
    for i in range(bordeMax,filas+bordeMax):
        for j in range(bordeMax,col+bordeMax):
            #recorremos ventanas
            for v in range(3,Smax+1,2):
                borde = Smax//2
                ventana = imagen_ampliada[i-borde:i+borde+1 , j-borde:j+borde+1]
                media = np.mean(ventana)
                desv = np.std(ventana)
                alpha = np.random.rand()
                inter1 = media - desv/2
                inter2 = media + desv/2
                res = res + inter1 + alpha*(inter1-inter2)
            res = res/borde
            imagen_filtrada[i-borde , j-borde] = res
    imagen_filtrada = np.uint8(imagen_filtrada)
    return imagen_filtrada
 

In [14]:
imagen = cv2.imread('arbol.jpg', cv2.IMREAD_GRAYSCALE)

# Aplicar el filtro no homogéneo
imagen2 =  ejercicio2(imagen,5)

# Mostrar la imagen original y la imagen filtrada
cv2.imshow('original', imagen)
cv2.imshow('ampliacion', imagen2)

# Esperar a que el usuario presione una tecla para cerrar las ventanas
cv2.waitKey(0)
cv2.destroyAllWindows()